<a href="https://colab.research.google.com/github/SpringBoard795/PicasoPhrase_Infosys_Internship_Nov2024/blob/MerajBegum/Hyperparameter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#BLEU Score and ROUGE Metrics Calculation

In [2]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=c102b12885a2e0503908de5dd827d34fd74c97bb01687ff2c4227aea9eb29c36
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [4]:
!pip install rouge

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge

# Load the file
file_path = '/content/drive/MyDrive/radiology/captions.txt'
with open(file_path, 'r') as file:
    lines = file.readlines()

# Assuming file contains lines of reference and generated captions:
# Example format: "reference_caption ||| generated_caption"
references = []
hypotheses = []
for line in lines:
    # Check if the delimiter is present before splitting
    if " ||| " in line:
        ref, hyp = line.strip().split(" ||| ", 1)  # Limit split to 1 to avoid issues with multiple delimiters
        references.append([ref.split()])
        hypotheses.append(hyp.split())
    else:
        # Handle lines without the delimiter (e.g., print a warning or skip)
        print(f"Warning: Skipping line '{line.strip()}' due to missing delimiter.")

if references and hypotheses:  # Ensure both lists have elements
    bleu_scores = [sentence_bleu(ref, hyp) for ref, hyp in zip(references, hypotheses)]
    average_bleu = sum(bleu_scores) / len(bleu_scores)
    print("Average BLEU Score:", average_bleu)
else:
    print("Warning: Cannot calculate BLEU score. 'references' or 'hypotheses' is empty.")

# Calculate ROUGE scores
# Check if references and hypotheses are empty before calculating the average ROUGE score
if references and hypotheses:  # Ensure both lists have elements
    rouge = Rouge()
    rouge_scores = [rouge.get_scores(" ".join(hyp), " ".join(ref[0]))[0] for ref, hyp in zip(references, hypotheses)]

    # Average ROUGE-L
    average_rouge_l = sum(score['rouge-l']['f'] for score in rouge_scores) / len(rouge_scores)
    print("Average ROUGE-L Score:", average_rouge_l)
else:
    print("Warning: Cannot calculate ROUGE score. 'references' or 'hypotheses' is empty.")

Streaming output truncated to the last 5000 lines.


#Hyperparameter Tuning with RandomSearchCV

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.datasets import make_classification

# Sample dataset
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

# Model and parameters
model = RandomForestClassifier(random_state=42)
param_dist = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Random search
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, n_iter=10, cv=3, random_state=42)
random_search.fit(X, y)

print("Best Parameters from RandomizedSearchCV:", random_search.best_params_)


Best Parameters from RandomizedSearchCV: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': None}


#Hyperparameter Tuning with GridSearchCV

In [25]:
from sklearn.model_selection import GridSearchCV

# Grid search parameters
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Grid search
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)
grid_search.fit(X, y)

print("Best Parameters from GridSearchCV:", grid_search.best_params_)


Best Parameters from GridSearchCV: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 50}


#Hyperparameter Tuning with Hyperopt

In [26]:
from hyperopt import hp, fmin, tpe, STATUS_OK
from sklearn.model_selection import cross_val_score

# Objective function
def objective(params):
    model = RandomForestClassifier(**params, random_state=42)
    score = cross_val_score(model, X, y, cv=3, scoring='accuracy').mean()
    return {'loss': -score, 'status': STATUS_OK}

# Define the search space
space = {
    'n_estimators': hp.choice('n_estimators', [50, 100, 200]),
    'max_depth': hp.choice('max_depth', [10, 20, 30, None]),
    'min_samples_split': hp.choice('min_samples_split', [2, 5, 10]),
    'min_samples_leaf': hp.choice('min_samples_leaf', [1, 2, 4])
}

# Hyperopt search
best_params = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=50)
print("Best Parameters from Hyperopt:", best_params)


100%|██████████| 50/50 [00:58<00:00,  1.18s/trial, best loss: -0.892979806153459]
Best Parameters from Hyperopt: {'max_depth': 2, 'min_samples_leaf': 0, 'min_samples_split': 1, 'n_estimators': 0}
